<div
  style="
    background-color: #f0f0f0;
    color:rgb(56, 56, 56);
    padding: 8px;
    display: flex;
    align-items: center;
    gap: 100px;
  "
>
  <img src="./images/brand.svg" style="max-height: 80px;">
  <strong>
    AI Saga: Data Science and Machine Learning</br>
    3.4.1. Finding the Minimal Neural Network for XOR
  </strong>
</div>

In [ ]:
# ⚠️ IMPORTANT NOTICE FOR STUDENTS ⚠️
#
# Please make sure to check the official instructions for this assignment in Canvas LMS
# as they may have been updated or changed. The instructions above are provided for
# reference only and may not reflect the most current requirements.
#
# Always refer to Canvas LMS for:
# - Latest assignment requirements
# - Due dates
# - Grading criteria
# - Any special instructions
#
# When in doubt, ask your instructor for clarification.
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import numpy as np
from rich.console import Console
from rich.table import Table

console = Console()

## XOR Problem

The XOR (exclusive OR) function is a classic problem in machine learning.
It is not linearly separable, meaning a single-layer network cannot solve it.
This notebook explores the minimal neural network architecture that can learn XOR consistently.

In [ ]:
inputs = torch.tensor(
    [[0.0, 0.0], [0.0, 1.0], [1.0, 0.0], [1.0, 1.0]], dtype=torch.float32
)
labels = torch.tensor([[0.0], [1.0], [1.0], [0.0]], dtype=torch.float32)

table = Table(title="XOR Truth Table")
table.add_column("Input 1", justify="center")
table.add_column("Input 2", justify="center")
table.add_column("Output", justify="center")

for input_row, label_row in zip(inputs, labels):
    table.add_row(
        str(int(input_row[0].item())),
        str(int(input_row[1].item())),
        str(int(label_row[0].item())),
    )

console.print(table)

## Helper Function: Train and Evaluate

This function trains any given model on the XOR dataset and returns the training losses.
We also count the number of trainable parameters to compare architectures.

In [ ]:
def count_parameters(model: nn.Module) -> int:
    """Count the total number of trainable parameters in a model."""
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def train_model(
    model: nn.Module,
    epochs: int = 3000,
    lr: float = 0.1,
    loss_fn: nn.Module = nn.MSELoss(),
) -> tuple[list[float], bool]:
    """Train the model on XOR data. Returns losses and whether it converged."""
    optimizer = optim.SGD(model.parameters(), lr=lr)
    losses = []
    converged = False

    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()
        output = model(inputs)
        loss = loss_fn(output, labels)
        loss.backward()
        optimizer.step()
        losses.append(loss.item())

        if loss.item() < 0.01:
            converged = True
            break

    return losses, converged


def check_predictions(model: nn.Module) -> bool:
    """Check if the model correctly predicts all XOR values."""
    model.eval()
    with torch.no_grad():
        preds = model(inputs)
        rounded = (preds > 0.5).float()
        return bool((rounded == labels).all().item())

## Architecture 1: Single Layer
A linear model cannot solve XOR because XOR is not linearly separable.
This is our baseline to demonstrate why a hidden layer is necessary.

In [ ]:
torch.manual_seed(42)

linear_model = nn.Sequential(
    nn.Linear(2, 1),
    nn.Sigmoid(),
)

linear_losses, linear_converged = train_model(linear_model, epochs=3000, lr=0.1)
linear_correct = check_predictions(linear_model)

print(f"Architecture: 2 -> 1")
print(f"Parameters: {count_parameters(linear_model)}")
print(f"Converged (loss < 0.01): {linear_converged}")
print(f"Correct predictions: {linear_correct}")
print(f"Final loss: {linear_losses[-1]:.4f}")

## Architecture 2: One Hidden Layer with 2 Neurons

The minimal hidden layer size theoretically needed to solve XOR is 2 neurons.
We test this as our first candidate for the minimal architecture.

In [ ]:
torch.manual_seed(42)

model_2h = nn.Sequential(
    nn.Linear(2, 2),
    nn.Sigmoid(),
    nn.Linear(2, 1),
    nn.Sigmoid(),
)

losses_2h, converged_2h = train_model(model_2h, epochs=3000, lr=0.5)
correct_2h = check_predictions(model_2h)

print(f"Architecture: 2 -> 2 -> 1")
print(f"Parameters: {count_parameters(model_2h)}")
print(f"Converged (loss < 0.01): {converged_2h}")
print(f"Correct predictions: {correct_2h}")
print(f"Final loss: {losses_2h[-1]:.4f}")

## Architecture 3: One Hidden Layer with 2 Neurons and Relu

We test the same minimal architecture but with ReLU activation instead of Sigmoid.
ReLU often converges faster due to less gradient saturation.

In [ ]:
torch.manual_seed(42)

model_relu = nn.Sequential(
    nn.Linear(2, 2),
    nn.ReLU(),
    nn.Linear(2, 1),
    nn.Sigmoid(),
)

losses_relu, converged_relu = train_model(model_relu, epochs=3000, lr=0.1)
correct_relu = check_predictions(model_relu)

print(f"Architecture: 2 -> 2 (ReLU) -> 1")
print(f"Parameters: {count_parameters(model_relu)}")
print(f"Converged (loss < 0.01): {converged_relu}")
print(f"Correct predictions: {correct_relu}")
print(f"Final loss: {losses_relu[-1]:.4f}")

## Architecture 4: One Hidden Layer with 3 Neurons

We increase the hidden layer to 3 neurons to see if consistency improves
at the cost of more parameters.

In [ ]:
torch.manual_seed(42)

model_3h = nn.Sequential(
    nn.Linear(2, 3),
    nn.Sigmoid(),
    nn.Linear(3, 1),
    nn.Sigmoid(),
)

losses_3h, converged_3h = train_model(model_3h, epochs=3000, lr=0.5)
correct_3h = check_predictions(model_3h)

print(f"Architecture: 2 -> 3 -> 1")
print(f"Parameters: {count_parameters(model_3h)}")
print(f"Converged (loss < 0.01): {converged_3h}")
print(f"Correct predictions: {correct_3h}")
print(f"Final loss: {losses_3h[-1]:.4f}")

## Consistency Test: Running Multiple Seeds

A single run may be lucky. We test each architecture over 20 different random seeds
to check if it converges consistently regardless of initialization.

In [ ]:
def test_consistency(
    architecture: str,
    build_fn,
    num_trials: int = 20,
    epochs: int = 3000,
    lr: float = 0.5,
) -> dict:
    """Test how consistently an architecture converges across random seeds."""
    successes = 0
    epochs_list = []

    for seed in range(num_trials):
        torch.manual_seed(seed)
        model = build_fn()
        losses, converged = train_model(model, epochs=epochs, lr=lr)
        correct = check_predictions(model)
        if converged and correct:
            successes += 1
            epochs_list.append(len(losses))

    avg_epochs = int(np.mean(epochs_list)) if epochs_list else epochs
    return {
        "architecture": architecture,
        "success_rate": successes / num_trials,
        "avg_epochs_to_converge": avg_epochs,
        "successes": successes,
        "trials": num_trials,
    }


results_linear = test_consistency(
    "2 -> 1 (no hidden)",
    lambda: nn.Sequential(nn.Linear(2, 1), nn.Sigmoid()),
    lr=0.1,
)

results_2h = test_consistency(
    "2 -> 2 -> 1 (Sigmoid)",
    lambda: nn.Sequential(nn.Linear(2, 2), nn.Sigmoid(), nn.Linear(2, 1), nn.Sigmoid()),
    lr=0.5,
)

results_relu = test_consistency(
    "2 -> 2 -> 1 (ReLU)",
    lambda: nn.Sequential(nn.Linear(2, 2), nn.ReLU(), nn.Linear(2, 1), nn.Sigmoid()),
    lr=0.1,
)

results_3h = test_consistency(
    "2 -> 3 -> 1 (Sigmoid)",
    lambda: nn.Sequential(nn.Linear(2, 3), nn.Sigmoid(), nn.Linear(3, 1), nn.Sigmoid()),
    lr=0.5,
)

all_results = [results_linear, results_2h, results_relu, results_3h]

for r in all_results:
    print(
        f"{r['architecture']}: {r['successes']}/{r['trials']} "
        f"({r['success_rate']*100:.0f}%) - Avg epochs: {r['avg_epochs_to_converge']}"
    )

## Comparison Table: Architectures by Parameters and Convergence

In [ ]:
param_counts = {
    "2 -> 1 (no hidden)": count_parameters(
        nn.Sequential(nn.Linear(2, 1), nn.Sigmoid())
    ),
    "2 -> 2 -> 1 (Sigmoid)": count_parameters(
        nn.Sequential(nn.Linear(2, 2), nn.Sigmoid(), nn.Linear(2, 1), nn.Sigmoid())
    ),
    "2 -> 2 -> 1 (ReLU)": count_parameters(
        nn.Sequential(nn.Linear(2, 2), nn.ReLU(), nn.Linear(2, 1), nn.Sigmoid())
    ),
    "2 -> 3 -> 1 (Sigmoid)": count_parameters(
        nn.Sequential(nn.Linear(2, 3), nn.Sigmoid(), nn.Linear(3, 1), nn.Sigmoid())
    ),
}

table = Table(title="Architecture Comparison")
table.add_column("Architecture", justify="left")
table.add_column("Parameters", justify="center")
table.add_column("Success Rate", justify="center")
table.add_column("Avg Epochs", justify="center")
table.add_column("Consistent?", justify="center")

for r in all_results:
    params = param_counts[r["architecture"]]
    consistent = "Yes" if r["success_rate"] >= 0.8 else "No"
    table.add_row(
        r["architecture"],
        str(params),
        f"{r['success_rate']*100:.0f}%",
        str(r["avg_epochs_to_converge"]),
        consistent,
    )

console.print(table)

## Training Curves for Successful Architectures

In [ ]:
torch.manual_seed(0)
m2h = nn.Sequential(nn.Linear(2, 2), nn.Sigmoid(), nn.Linear(2, 1), nn.Sigmoid())
l2h, _ = train_model(m2h, epochs=3000, lr=0.5)

torch.manual_seed(0)
mrelu = nn.Sequential(nn.Linear(2, 2), nn.ReLU(), nn.Linear(2, 1), nn.Sigmoid())
lrelu, _ = train_model(mrelu, epochs=3000, lr=0.1)

torch.manual_seed(0)
m3h = nn.Sequential(nn.Linear(2, 3), nn.Sigmoid(), nn.Linear(3, 1), nn.Sigmoid())
l3h, _ = train_model(m3h, epochs=3000, lr=0.5)

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(l2h, label="2->2->1 (Sigmoid)")
plt.plot(lrelu, label="2->2->1 (ReLU)")
plt.plot(l3h, label="2->3->1 (Sigmoid)")
plt.xlabel("Epoch")
plt.ylabel("Loss (MSE)")
plt.title("Training Curves - All Architectures")
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(l2h[:500], label="2->2->1 (Sigmoid)")
plt.plot(lrelu[:500], label="2->2->1 (ReLU)")
plt.plot(l3h[:500], label="2->3->1 (Sigmoid)")
plt.xlabel("Epoch")
plt.ylabel("Loss (MSE)")
plt.title("Training Curves - First 500 Epochs")
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

## Scientific Foundation: Why XOR Requires Non-Linearity

The XOR function produces output 1 only when inputs differ (0,1) or (1,0), and output 0 when inputs are the same (0,0) or (1,1).

**Linear separability:** A problem is linearly separable if a single hyperplane (line in 2D) can separate the two classes.
XOR is **not** linearly separable — no single straight line can correctly classify all four XOR points into two groups.

**Why a hidden layer solves this:**
- Each neuron in the hidden layer learns a linear boundary
- With 2 hidden neurons, the network creates 2 linear boundaries
- The output neuron combines these boundaries non-linearly
- This composition of linear transformations with non-linear activations creates a non-linear decision boundary

**The minimum theoretical requirement:**
- At least 1 hidden layer is needed (to introduce non-linearity)
- At least 2 neurons in that hidden layer (to create 2 boundaries that define XOR regions)
- This gives us the architecture: `2 → 2 → 1` with 9 parameters as the theoretical minimum

## Final Conclusion

### Architecture Trade-off

There is a clear trade-off between **parameter efficiency** and **convergence consistency**:

- **Fewer parameters** (`2 → 2 → 1`, 9 params): Minimal model, but sensitive to random initialization. Not every seed converges within 3000 epochs.
- **More parameters** (`2 → 3 → 1`, 13 params): More reliable convergence because additional neurons provide more optimization paths, but at the cost of model complexity.

### Scientific Conclusion

The XOR problem demonstrates a fundamental principle in neural networks: **non-linearity is essential for solving non-linearly separable problems**. 
The single-layer model (3 parameters) completely fails because no linear boundary can separate XOR outputs. 
Adding one hidden layer with just 2 neurons (9 parameters total) provides the minimal non-linear capacity needed. 
However, achieving **consistent** convergence (independent of initialization) requires either:
1. A slightly larger architecture (`2 → 3 → 1`, 13 parameters), or
2. Careful tuning of the learning rate (lr=0.5 works best for Sigmoid with 2 hidden neurons)

**The answer to the trade-off question:** Yes, there is a clear trade-off. The minimal architecture (`2 → 2 → 1`) solves XOR but is not perfectly consistent. 
The next-smallest architecture (`2 → 3 → 1`) adds only 4 parameters but significantly improves consistency. 
This suggests that the **practical minimum** for consistent XOR learning is `2 → 3 → 1` with 13 parameters, 
while the **theoretical minimum** is `2 → 2 → 1` with 9 parameters.